## Overfitting and Underfitting Analysis

### 1. Motivation 

The primary goal of this project is to examine whether incorporating neighbourhood-based statistical features can improve house price prediction performance.

However, evaluating model quality using **Test Mean Squared Error (MSE) alone is insufficient**.  
A lower Test MSE may arise from two different causes:

1. genuine improvement in model generalisation, or  
2. overfitting, where the model memorises training data but fails to generalise.

To distinguish between these possibilities, we compare **Train MSE and Test MSE** for each model.  
The relative difference between these two metrics provides an indication of the model's generalisation behaviour.


### 2. Diagnostic Method 

We define the **Test-to-Train MSE Ratio** as a simple diagnostic indicator:

Ratio = Test MSE / Train MSE

Diagnosis is based on **relative comparison rather than absolute error thresholds**,  
since the absolute magnitude of MSE depends on the scale of the dataset.

| Condition | Interpretation |
|---|---|
| Ratio > 1.5 | Possible overfitting — test error substantially larger than training error |
| Ratio < 1.1 | Small generalisation gap — if both errors remain high, this may indicate underfitting |
| Otherwise | Reasonable fit — no large train–test gap observed |

The value **1.5** is used as a heuristic diagnostic threshold.  
If the test error exceeds the training error by more than approximately 50%, the model may exhibit signs of overfitting.

In [ ]:
# ============================================================
# Overfitting / Underfitting Check
#
# Motivation:
#   Reporting Test MSE alone is insufficient to evaluate model quality.
#   A lower Test MSE could result from genuine generalisation improvement,
#   or simply from the model overfitting the training data.
#   By comparing Train MSE and Test MSE, we can assess whether the
#   improvement in Model 2 is more likely to reflect better generalisation
#   rather than a larger train–test gap.
#
# Why Train vs Test (not Train vs Val):
#   In the final training stage, the training and validation sets are
#   merged together to maximise the amount of data available for training.
#   This means no independent validation set remains.
#   The held-out test set therefore serves as the reference for
#   evaluating generalisation performance.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

def diagnose_final(model_name, train_mse, test_mse, threshold_ratio=1.5):  # heuristic threshold
    """
    Print an overfitting/underfitting diagnosis based on Test/Train MSE ratio.

    Diagnosis rules:
        ratio > 1.5  → Possible overfitting
        ratio < 1.1  → Low generalisation gap; possible underfitting if both errors remain high
        Otherwise    → Reasonable fit
    """
    # Compute ratio; set to infinity if train_mse is zero to avoid division by zero
    ratio = test_mse / train_mse if train_mse > 0 else float('inf')

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Train MSE       : {train_mse:,.4f}")
    print(f"  Test  MSE       : {test_mse:,.4f}")
    print(f"  Test/Train Ratio: {ratio:.3f}")

    # Diagnosis uses relative comparison rather than absolute thresholds,
    # as the appropriate MSE scale is dataset-dependent.
    if ratio > threshold_ratio:
        # Test MSE notably larger than Train MSE → possible overfitting
        print("  ⚠ Diagnosis : POSSIBLE OVERFITTING — test MSE is notably larger than train MSE")
    elif ratio < 1.1:
        # Train and test MSE are similar but no absolute threshold is used,
        # as MSE scale is dataset-dependent.
        print("  ~ Diagnosis : LOW GENERALISATION GAP — train and test errors are similar")
    else:
        # Train and test MSE are reasonably close → no large generalisation gap
        print("  ✓ Diagnosis : REASONABLE FIT — no large train–test gap is observed")
    print(f"{'='*55}")

### 3. Results 

| Model | Train MSE | Test MSE | Ratio | Diagnosis |
|---|---|---|---|---|
| Model 1 — Baseline | 0.2398 | 0.2485 | 1.036 | ✓ Reasonable Fit |
| Model 2 — Enhanced | 0.1557 | 0.1805 | 1.159 | ✓ Reasonable Fit |

Both models exhibit **relatively small train–test gaps**, suggesting that neither model shows strong evidence of overfitting or severe underfitting.


Model 2 achieves a **lower Test MSE** than Model 1, suggesting that the addition of neighbourhood-based features improves predictive performance.。

Although Model 2 shows a slightly larger Test/Train ratio, the value remains well below the overfitting threshold, indicating that the improvement is unlikely to be caused by excessive memorisation of the training data.


Overall, the results suggest that neighbourhood-based features contribute to improved generalisation performance without introducing clear signs of overfitting.

In [ ]:
# ============================================================
# MODEL 1 — Final Baseline
#
# baseline_final_model was trained on the merged train+val set
# using only the original features, with no neighbourhood features.
# baseline_train_mse and baseline_test_mse_final were already computed
# immediately after training, so we reuse them directly here.
# ============================================================

m1_train_mse = baseline_train_mse        # already computed above
m1_test_mse  = baseline_test_mse_final   # already computed above

diagnose_final("Model 1 — Final Baseline", m1_train_mse, m1_test_mse)



# ============================================================
# MODEL 2 — Final Enhanced
#
# model_enh_final was trained on the merged train+val set
# with neighbourhood-based price statistics added as extra inputs.
# enhanced_test_mse_final was already computed immediately after training.
# Train MSE was not computed in the original code, so we compute it
# once here using the already-fitted model. No re-training is required.
# ============================================================

m2_test_mse  = enhanced_test_mse_final   # already computed above

# Train MSE for Model 2 was not computed in the original code.
# Computed once here using the already-fitted model_enh_final.
m2_train_mse = mean_squared_error(        # not in original code, compute once here
    y_train_final,
    model_enh_final.predict(X_train_enh_final)
)

diagnose_final("Model 2 — Final Enhanced", m2_train_mse, m2_test_mse)

### Figure Caption 

**Figure 1 — Train vs Test MSE Comparison**

Bar chart comparing the training and test mean squared errors of the two final models.  
The gap between Train MSE and Test MSE provides a visual indication of potential overfitting: a small gap suggests better generalisation beyond the training data.

**Figure 2 — Approximate Learning Curves**

Approximate learning curves showing how the training MSE evolves during optimisation.  
The solid line represents the training error trajectory, while the dashed horizontal line indicates the final test error as a reference.

These curves provide a qualitative view of the optimisation behaviour rather than exact epoch-level training logs.



### 4. Learning Curve Analysis 

In addition to the final MSE comparison, **approximate learning curves** are plotted to visualise how the training error evolves during optimisation.

The curves are generated by **re-simulating training from scratch** using `warm_start=True` with `max_iter=1` in each call to `fit()`.

Although the architecture, hyperparameters, and random seed are identical to those of the final models, this procedure **does not reproduce the exact training history**.

In sklearn's `MLPRegressor`, each call to `fit(max_iter=1)` **does not strictly correspond to one epoch**, due to optimiser internal states and stochastic data shuffling.

Therefore, the curves should be interpreted as **approximate optimisation trajectories**.

Only the **training error** is recorded per iteration, while the final **test error** is shown as a horizontal reference line.  
This avoids repeatedly evaluating the test set during training and helps prevent methodological bias.

In [ ]:
# ============================================================
# VISUALISATION 1 — Bar chart: Train vs Test MSE
#
# Purpose:
#   Visually compare the gap between Train MSE and Test MSE for both
#   models. A larger gap may indicate possible overfitting.
# ============================================================

labels     = ['Model 1\n(Baseline)', 'Model 2\n(Enhanced)']
train_mses = [m1_train_mse, m2_train_mse]
test_mses  = [m1_test_mse,  m2_test_mse]

# X-axis positions for each group of bars
x     = np.arange(len(labels))
width = 0.35  # Width of each individual bar

fig, ax = plt.subplots(figsize=(8, 5))

# Plot Train MSE bars on the left of each group
bars1 = ax.bar(x - width/2, train_mses, width, label='Train MSE', color='steelblue',  alpha=0.85)

# Plot Test MSE bars on the right of each group
bars2 = ax.bar(x + width/2, test_mses,  width, label='Test MSE',  color='darkorange', alpha=0.85)

ax.set_ylabel('MSE')
ax.set_title('Train vs Test MSE — Model 1 and Model 2')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Annotate each bar with its exact MSE value
# We use ax.text() instead of bar_label() for compatibility
# with older versions of matplotlib
for bar, val in zip(bars1, train_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, test_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALISATION 2 — Approximate Learning Curves (Train MSE vs Epoch)
#
# Purpose:
#   The ratio test above evaluates only the final model state.
#   To provide additional insight, approximate learning curves
#   are plotted to illustrate how Train MSE evolves during training
#   under the same architecture and hyperparameters as the final models.
#
# Methodological note:
#   Training is re-simulated using warm_start=True with max_iter=1
#   per fit() call. This approximates the optimisation trajectory
#   but does NOT reproduce the exact training history of the final model.
# ============================================================


def collect_train_curve(Xtr, ytr, hidden, act, max_iter=800):
    """
    Simulate training with warm_start and record Train MSE at each step.

    This produces an approximate learning curve rather than the
    exact optimisation path of the final fitted model.

    The test set is NOT evaluated per epoch to avoid using it
    as a model selection reference.

    Returns:
        list: training MSE values recorded after each step
    """
    # warm_start=True retains weights between calls so training is continuous
    m = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation=act,
        solver='adam',
        learning_rate_init=1e-3,
        max_iter=1,        # Each call performs a short training step 
        warm_start=True,   # Continue from previous weights 
        early_stopping=False,
        random_state=26
    )
    
    # Design choice: only Train MSE is plotted per step.
    # Test MSE is shown as a single horizontal reference line
    # to avoid using the test set for model selection during training.
    train_curve = []
    for _ in range(max_iter):
        m.fit(Xtr, ytr)  # Train for one more epoch 

        # Record Train MSE after this epoch
        pred = m.predict(Xtr)
        train_curve.append(mean_squared_error(ytr, pred))
    return train_curve

### 5. Interpreting Learning Curves 

| Pattern | Interpretation |
|---|---|
| Train curve converges near test reference line | May indicate good generalisation |
| Train curve falls far below test reference line | May indicate potential overfitting |
| Both train curve and test line remain high | May indicate possible underfitting |

These interpretations should be treated as **qualitative indicators** rather than definitive diagnostic criteria.

This is because no independent validation trajectory is available and the learning curves themselves are approximations.

In [ ]:
print("\nCollecting learning curves (this may take a moment)…")

# Collect approximate train curve for Model 1 — baseline features only
lc_train_m1 = collect_train_curve(
    X_train_final_base, y_train_final_base,
    hidden=(128, 32), act='tanh'
)

# Collect approximate train curve for Model 2 — with neighbourhood features
lc_train_m2 = collect_train_curve(
    X_train_enh_final, y_train_final,
    hidden=(128, 32), act='tanh'
)

# Plot approximate learning curves for both models side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lc_tr, final_test_mse, title in zip(
    axes,
    [lc_train_m1, lc_train_m2],
    [m1_test_mse,  m2_test_mse],
    ['Model 1 — Final Baseline', 'Model 2 — Final Enhanced']
):
    epochs = range(1, len(lc_tr) + 1)

    # Solid blue line = Train MSE per epoch (approximate)
    ax.plot(epochs, lc_tr, label='Train MSE (approx.)', color='steelblue')

    # Horizontal dashed orange line = final Test MSE (reference only)
    # The test set is evaluated only once at the end, not per epoch
    ax.axhline(
        y=final_test_mse,
        color='darkorange',
        linestyle='--',
        linewidth=1.2,
        label=f'Final Test MSE = {final_test_mse:.4f}'
    )

    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle(
    'Approximate Learning Curves — Overfitting / Underfitting Check',
    fontsize=13
)
plt.tight_layout()
plt.show()

### 6. Final Discussion

The analysis suggests that incorporating neighbourhood-based statistical features  
improves predictive performance while maintaining a reasonable level of generalisation.

Compared with the baseline model, the enhanced model achieves a lower test error  
without introducing a large train–test gap, which indicates that the improvement  
is unlikely to be caused by excessive memorisation of the training data.

Overall, the results provide empirical evidence that neighbourhood information  
can serve as a useful feature for improving house price prediction models.